In [2]:
from rabtracks import *
%load_ext line_profiler

# IBTrACS

In [3]:
def prepare_ibtracs(b, outfile):
    # Load the entire ibtracs dataset
    ib=huracanpy.load(source="ibtracs", ibtracs_subset=b, baselon = -180)
    # Rename Track IDs
    ib = ib.rename({"sid":"track_id"})
    # Keep only 6-hourly points
    six_hours_ns = np.timedelta64(6, "h").astype("timedelta64[ns]").astype("int64")  # 6h in nanoseconds
    ib = ib.where(ib.time.astype("datetime64[ns]").astype("int64") % six_hours_ns == 0, drop = True)
    # Remove empty variables
    null = (ib.isnull().sum() == len(ib.record))
    null_vars = [v for v in list(ib.variables.keys()) if null[v]]
    ib = ib.drop_vars(null_vars)
    # Convert knots to m/s
    wind_vars=[v for v in list(ib.variables.keys()) if v.endswith("_wind")] + [v for v in list(ib.variables.keys()) if v.endswith("_gust")] + [v for v in list(ib.variables.keys()) if v.endswith("_speed")]
    for v in wind_vars:
        ib[v] = ib[v] * 0.514444
        ib[v].attrs["units"] = "ms-1"
    # Convert nmile to km
    nmile_vars = ["usa_r34_ne", "usa_r34_se", "usa_r34_sw", "usa_r34_nw", "usa_r50_ne", "usa_r50_se", "usa_r50_sw", "usa_r50_nw", "usa_r64_ne", "usa_r64_se", "usa_r64_sw", "usa_r64_nw", "usa_roci", "usa_rmw", "usa_eye", 
                       "tokyo_r50_long", "tokyo_r50_short", "tokyo_r30_long", "tokyo_r30_short", 
                       "reunion_rmw", "reunion_r34_ne", "reunion_r34_se", "reunion_r34_sw", "reunion_r34_nw", "reunion_r50_ne", "reunion_r50_se", "reunion_r50_sw", "reunion_r50_nw", "reunion_r64_ne", "reunion_r64_se", "reunion_r64_sw", "reunion_r64_nw", 
                       "bom_rmw", "bom_r34_ne", "bom_r34_se", "bom_r34_sw", "bom_r34_nw", "bom_r50_ne", "bom_r50_se", "bom_r50_sw", "bom_r50_nw", "bom_r64_ne", "bom_r64_se", "bom_r64_sw", "bom_r64_nw", "bom_roci", "bom_eye", 
                       "td9635_roci", "usa_searad_ne", "usa_searad_se", "usa_searad_sw", "usa_searad_nw"]
    for v in nmile_vars:
        if v in list(ib.variables.keys()):
            ib[v] = ib[v] * 1.60934
            ib[v].attrs["units"] = "km"
    # Convert ft to m
    ft_vars = ["usa_seahgt"]
    for v in ft_vars:
        if v in list(ib.variables.keys()):
            ib[v] = ib[v] * 0.3048
            ib[v].attrs["units"] = "m"
    # Save the dataset
    with open(outfile, "wb") as f:
        pkl.dump(ib, f)
    with lzma.open(outfile+".xz", "wb") as f:
        pkl.dump(ib, f)
    del ib
    
for b in tqdm(BASINS):
    outfile = "ibtracs/ibtracs_"+b+".pkl"
    if not os.path.isfile(outfile):
        prepare_ibtracs(b, outfile)

100%|██████████| 7/7 [00:00<00:00, 5213.09it/s]


# Reanalysis tracks

In [4]:
def remove_unused_vars(ds, 
                       vars2remove = [
        "full_name", "short_label", "transition_zone", "track_info", "i", "j", "zs", # SyCLoPS extra vars
         "basin", 'wind925_lon','wind925_lat', 'wind925', 'wind_speed_925', 'wind_speed_10m', 'longitude_vo850', 'latitude_vo850',# TRACK extra vars
                        ]):
    varlist = list(ds.variables.keys()) # List of variables in ds
    extravars = list(set(varlist) & set(vars2remove)) # List of variables to remove from ds
    return ds.drop_vars(extravars)

In [5]:
def rename_vars(ds, rename_dict = {"psl":"pres", "adjusted_label":"label", "tropical_flag":"is_tc", 
                                   "Vtl":"vtl", "Vtu":"vtu", "B":"b",
                                  "longitude_psl":"lon_pres", "latitude_psl":"lat_pres", "psl_lon":"lon_pres", "psl_lat":"lat_pres",
                                  'longitude_wind10':"lon_wind", 'latitude_wind10':"lat_wind", 'wind10_lon':"lon_wind",'wind10_lat':"lat_wind",}):

    for n in rename_dict:
        if n in list(ds.variables.keys()):
            ds = ds.rename({n:rename_dict[n]})
    return ds

In [6]:
def reduce_str_length(ds, strvars=["track_id", "label"]):
    varlist = list(ds.variables.keys()) # List of variables in ds
    strvars = list(set(varlist) & set(strvars)) # List of str vars in ds
    for var in strvars:
        max_str_length = str(ds[var].astype(str).str.len().max().values)
        ds[var] = ds[var].astype("<S"+max_str_length)
    return ds

In [7]:
def reduce_float_to_single(ds, 
                           floatvars = [
    "lon", "lat", "pres", "wind10", "vo850","vtl", "vtu", "b",
                            ]):
    varlist = list(ds.variables.keys()) # List of variables in ds
    floatvars = list(set(varlist) & set(floatvars)) # List of float vars in ds
    for var in floatvars:
        ds[var] = ds[var].astype(np.float32)
    return ds

In [8]:
def reduce_int_to_int32(ds, intvars = ["lpsarea", "ike",]):
    varlist = list(ds.variables.keys()) # List of variables in ds
    intvars = list(set(varlist) & set(intvars)) # List of int vars in ds
    for var in intvars:
        ds[var] = ds[var].astype(np.int32)
    return ds

In [9]:
def convert_flags_to_bool(ds, flagvars = ["is_tc"]):
    varlist = list(ds.variables.keys()) # List of variables in ds
    flagvars = list(set(varlist) & set(flagvars)) # List of flag vars in ds
    for var in flagvars:
        ds[var] = ds[var].astype(bool)
    return ds

## SyCLoPS

In [10]:
# Find all raw data files
flist = glob("raw_data/SyCLoPS/SyCLoPS_classified*.parquet")
# List of corresponding sources
syclops_sources = [f.split("_")[-1].split(".")[0] for f in flist]
# Path dict
filepaths = {syclops_sources[i]:flist[i] for i in range(len(flist))}

In [16]:
# Load, treat and save the data
def prepare_SyCLoPS(s, infile, outfile):
    syclops_data = huracanpy.load(infile, baselon = -180).rename({"tid":"track_id", "mslp":"pres", "ws":"wind10"})
    syclops_data = rename_vars(syclops_data)
    # Convert pressures to hPa
    syclops_data = syclops_data.assign(pres = syclops_data.pres / 100)
    # Subset 6-hourly data
    syclops_data = syclops_data.where(syclops_data.time.dt.hour % 6 == 0, drop = True)
    # Reduce size 
    syclops_data = remove_unused_vars(syclops_data)
    syclops_data = reduce_str_length(syclops_data)
    syclops_data = reduce_float_to_single(syclops_data)
    syclops_data = reduce_int_to_int32(syclops_data)
    syclops_data = convert_flags_to_bool(syclops_data)
    # Save
    save_xarray_to_pickle(syclops_data, outfile, compress=True)
    del syclops_data

for s in tqdm(syclops_sources):
    outfile = "RA_tracks/SyCLoPS-"+s+".pkl"
    if not os.path.isfile(outfile):
        prepare_SyCLoPS(s, filepaths[s], outfile)

100%|██████████| 3/3 [02:43<00:00, 54.40s/it]


## TRACK (all tracks without tcident, CPS or WCSI)

In [17]:
flist = glob("raw_data/TRACK_netcdf/*/*TRACK_all.nc")
# List of available datasets
track_sources = [f.split('/')[2] for f in flist]
# Path dict
filepaths = {track_sources[i]:flist[i] for i in range(len(flist))}
filepaths

{'JRA3Q': 'raw_data/TRACK_netcdf/JRA3Q/JRA3Q_TRACK_all.nc',
 'ERA5': 'raw_data/TRACK_netcdf/ERA5/ERA5_TRACK_all.nc',
 'ECMWF-OP-AN': 'raw_data/TRACK_netcdf/ECMWF-OP-AN/ECMWF-OP-AN_TRACK_all.nc',
 'MERRA2': 'raw_data/TRACK_netcdf/MERRA2/MERRA2_TRACK_all.nc',
 'NCEP': 'raw_data/TRACK_netcdf/NCEP/NCEP_TRACK_all.nc'}

In [18]:
# Load, treat and save the data
def prepare_TRACK(s, infile, outfile):
    track_data = xr.open_dataset(infile)
    track_data = rename_vars(track_data)
    # Reduce size 
    track_data = remove_unused_vars(track_data)
    track_data = reduce_str_length(track_data)
    track_data = reduce_float_to_single(track_data)
    track_data = reduce_int_to_int32(track_data)
    track_data = convert_flags_to_bool(track_data)
    print(s, list(track_data.variables.keys()))
    # Save
    save_xarray_to_pickle(track_data, outfile, compress=True)
    del track_data

for s in tqdm(track_sources):
    outfile = "RA_tracks/TRACK-"+s+".pkl"
    if not os.path.isfile(outfile):
        prepare_TRACK(s, filepaths[s], outfile)

  0%|          | 0/5 [00:00<?, ?it/s]

JRA3Q ['lon', 'lat', 'vo850', 'lon_pres', 'lat_pres', 'pres', 'lon_wind', 'lat_wind', 'wind10', 'track_id', 'time']


 20%|██        | 1/5 [05:41<22:47, 341.77s/it]

ERA5 ['lon', 'lat', 'vo850', 'lon_pres', 'lat_pres', 'pres', 'lon_wind', 'lat_wind', 'wind10', 'track_id', 'time']


 40%|████      | 2/5 [12:17<18:40, 373.46s/it]

...

 60%|██████    | 3/5 [13:34<07:56, 238.23s/it]

MERRA2 ['lon', 'lat', 'vo850', 'lon_pres', 'lat_pres', 'pres', 'lon_wind', 'lat_wind', 'wind10', 'track_id', 'time']


 80%|████████  | 4/5 [17:12<03:50, 230.18s/it]

NCEP ['lon', 'lat', 'vo850', 'lon_pres', 'lat_pres', 'pres', 'lon_wind', 'lat_wind', 'wind10', 'track_id', 'time']


100%|██████████| 5/5 [20:25<00:00, 245.13s/it]


## TRACK with CPS information

In [19]:
flist = glob("raw_data/TRACK_netcdf/*/*TRACK_tcident_nolat_CPS.nc") 
# List of available datasets
cps_sources = [f.split('/')[2] for f in flist]
# Path dict
filepaths = {cps_sources[i]:flist[i] for i in range(len(flist))}
filepaths

{'JRA3Q': 'raw_data/TRACK_netcdf/JRA3Q/JRA3Q_TRACK_tcident_nolat_CPS.nc',
 'ERA5': 'raw_data/TRACK_netcdf/ERA5/ERA5_TRACK_tcident_nolat_CPS.nc'}

In [20]:
# Load, treat and save the data
def prepare_TRACK_CPS(s, infile, outfile):
    track_data = xr.open_dataset(infile)
    track_data = rename_vars(track_data)
    # Reduce size 
    track_data = remove_unused_vars(track_data)
    track_data = remove_unused_vars(track_data, ["pres"])
    track_data = reduce_str_length(track_data)
    track_data = reduce_float_to_single(track_data)
    track_data = reduce_int_to_int32(track_data)
    track_data = convert_flags_to_bool(track_data)
    print(list(track_data.variables.keys()))
    # Save
    save_xarray_to_pickle(track_data, outfile, compress=True)
    del track_data

for s in tqdm(cps_sources):
    outfile = "RA_tracks/TRACK_CPS-"+s+".pkl"
    if not os.path.isfile(outfile):
        prepare_TRACK_CPS(s, filepaths[s], outfile)

  0%|          | 0/2 [00:00<?, ?it/s]

['lon', 'lat', 'vtl', 'vtu', 'b', 'track_id', 'time']


 50%|█████     | 1/2 [00:16<00:16, 16.88s/it]

['lon', 'lat', 'vtl', 'vtu', 'b', 'track_id', 'time']


100%|██████████| 2/2 [00:22<00:00, 11.32s/it]


## TRACK with WCSI flags

In [21]:
# Tracks with WCSI labels
wcsi_sources = ["ERA5", "JRA3Q"]
# Path dict
filepaths = {s: glob("raw_data/TRACK_subsets/"+s+"/"+s+"*_WCSI.nc")[-1] for s in wcsi_sources}
filepaths

{'ERA5': 'raw_data/TRACK_subsets/ERA5/ERA5_WCSI.nc',
 'JRA3Q': 'raw_data/TRACK_subsets/JRA3Q/JRA3Q_nolat-tcident_WCSI.nc'}

In [22]:
# Load, treat and save the data
def prepare_TRACK_WCSI(s, infile, outfile):
    track_data = huracanpy.load(infile, baselon = -180)#[["lon", "lat", "time", "is_tc"]]
    track_data = rename_vars(track_data)
    track_data = rename_vars(track_data, {"is_tc":"WCSI"})
    # Reduce size 
    track_data = remove_unused_vars(track_data)
    track_data = reduce_str_length(track_data)
    track_data = reduce_float_to_single(track_data)
    track_data = reduce_int_to_int32(track_data)
    track_data = convert_flags_to_bool(track_data)
    print(list(track_data.variables.keys()))
    # Save
    save_xarray_to_pickle(track_data, outfile, compress=True)
    return track_data

for s in tqdm(wcsi_sources):
    outfile = "RA_tracks/TRACK_WCSI-"+s+".pkl"
    if not os.path.isfile(outfile):
        prepare_TRACK_WCSI(s, filepaths[s], outfile)

  0%|          | 0/2 [00:00<?, ?it/s]

['time', 'lon', 'lat', 'vorticity', 'mslp_lon', 'mslp_lat', 'mslp', 'vmax925hpa_lon', 'vmax925hpa_lat', 'vmax925hpa', 'vmax10m_lon', 'vmax10m_lat', 'vmax10m', 'cps_vtl', 'cps_vtu', 'cps_b', 'track_id_original', 'relative_vorticity', 'relative_vorticity_lon', 'relative_vorticity_lat', 'WCSI', 'pressure', 'track_id']


 50%|█████     | 1/2 [00:22<00:22, 22.30s/it]

['time', 'lon', 'lat', 'vorticity', 'mslp_lon', 'mslp_lat', 'mslp', 'vmax925hpa_lon', 'vmax925hpa_lat', 'vmax925hpa', 'vmax10m_lon', 'vmax10m_lat', 'vmax10m', 'cps_vtl', 'cps_vtu', 'cps_b', 'track_id_original', 'relative_vorticity', 'relative_vorticity_lon', 'relative_vorticity_lat', 'WCSI', 'pressure', 'track_id']


100%|██████████| 2/2 [00:45<00:00, 22.82s/it]
